Celda 1. Imports y configuracion general

In [7]:
import os
import gc
import warnings
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import time
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

warnings.filterwarnings("ignore")

CARPETA_MODELO = "modelo_guardado"
INPUT_FILE = "fused_it_ot.csv"
OUTPUT_ALERTS = "alertas_ids.csv"
OUTPUT_RESULTS = "resultados_ids.csv"

# =========================
# MEMORIA
# =========================
CHUNK_SIZE = 50000
BATCH_PRED = 4096

# =========================
# SEMILLA
# =========================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Configuracion inicial cargada.")

Configuracion inicial cargada.


Celda 2. Carga de modelos preentrenados

In [ ]:
ruta_paquete = os.path.join(CARPETA_MODELO, "Modelo_hibrido.pkl")

paquete_modelo = joblib.load(ruta_paquete)

xgb = paquete_modelo["xgb"]
lgbm = paquete_modelo["lgbm"]
meta_learner = paquete_modelo["meta_learner"]
meta_scaler = paquete_modelo["meta_scaler"]
scaler = paquete_modelo["scaler"]
selector = paquete_modelo["selector"]
use_pca = paquete_modelo.get("use_pca", False)
pca = paquete_modelo["pca"] if use_pca else None
use_autoencoder = paquete_modelo.get("use_autoencoder", False)
ae_threshold = paquete_modelo.get("ae_threshold", None)

feature_names = paquete_modelo.get("feature_names", None)
if feature_names is None:
    ruta_feature_names = os.path.join(CARPETA_MODELO, "feature_names.pkl")
    if os.path.exists(ruta_feature_names):
        feature_names = joblib.load(ruta_feature_names)

if feature_names is None:
    raise ValueError("No se encontro feature_names. Debes guardarlo desde el entrenamiento.")

print("Modelo hibrido cargado.")
print("use_pca:", use_pca)
print("use_autoencoder:", use_autoencoder)
print("Numero de features esperadas:", len(feature_names))

final_dnn = tf.keras.models.load_model(os.path.join(CARPETA_MODELO, "dnn_model.keras"))
print("DNN cargada.")

autoencoder = None
ae_info = None

if use_autoencoder:
    ruta_ae = os.path.join(CARPETA_MODELO, "autoencoder.keras")
    ruta_ae_info = os.path.join(CARPETA_MODELO, "autoencoder_info.pkl")

    if os.path.exists(ruta_ae) and os.path.exists(ruta_ae_info):
        autoencoder = tf.keras.models.load_model(ruta_ae)
        ae_info = joblib.load(ruta_ae_info)
        ae_threshold = ae_info.get("threshold", ae_threshold)
        print("Autoencoder cargado.")
        print("AE threshold:", ae_threshold)
    else:
        print("Advertencia: no se encontro autoencoder.keras o autoencoder_info.pkl")
        use_autoencoder = False

ruta_encoders = os.path.join(CARPETA_MODELO, "encoders.pkl")
if not os.path.exists(ruta_encoders):
    raise FileNotFoundError(
        "Falta encoders.pkl. Debes guardarlo en el entrenamiento para usar exactamente la misma codificacion."
    )

encoders = joblib.load(ruta_encoders)
print("Encoders cargados.")

Modelo hibrido cargado.
use_pca: True
use_autoencoder: True
Numero de features esperadas: 19
DNN cargada.
Autoencoder cargado.
AE threshold: 0.17674311995506287
Encoders cargados.


Celda 3. Funciones de apoyo

In [ ]:
def downcast_numeric(df_in: pd.DataFrame) -> pd.DataFrame:
    df_out = df_in.copy()
    for col in df_out.columns:
        if pd.api.types.is_float_dtype(df_out[col]):
            df_out[col] = pd.to_numeric(df_out[col], downcast="float")
        elif pd.api.types.is_integer_dtype(df_out[col]):
            df_out[col] = pd.to_numeric(df_out[col], downcast="integer")
    return df_out


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip().str.lower()
    return df


def safe_map_encoder(series: pd.Series, encoder, fallback_value: int = 0) -> pd.Series:
    """
    Usa el LabelEncoder del entrenamiento.
    Si aparece una categoria no vista, la manda al fallback_value.
    """
    mapping = {cls: idx for idx, cls in enumerate(encoder.classes_)}
    return (
        series.fillna("unknown")
        .astype(str)
        .str.strip()
        .str.lower()
        .map(lambda x: mapping.get(x, fallback_value))
        .astype("int32")
    )


def preprocess_fused_it_ot_chunk(chunk: pd.DataFrame):
    """
    Convierte fused_it_ot.csv al espacio exacto que espera el modelo.
    Elimina trafico normal y conserva IP y OT solo como metadata para alertas.
    """
    chunk = normalize_columns(chunk)
    if "traffic_label" in chunk.columns:
        tl = (
            chunk["traffic_label"]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        normal_mask = tl.isin(["0", "normal", "normal_pcaps"])
        chunk = chunk.loc[~normal_mask].copy()

    # Si no queda nada, salir
    if len(chunk) == 0:
        return None, None, None
    meta_cols = [
        c for c in [
            "timestamp", "src_ip", "dst_ip", "device_type",
            "application_name", "application_category",
            "attack_type", "traffic_label", "raw_log"
        ]
        if c in chunk.columns
    ]
    meta = chunk[meta_cols].copy() if meta_cols else pd.DataFrame(index=chunk.index)

    # Etiqueta real, ya filtrada
    y_true = None
    if "traffic_label" in chunk.columns:
        y_true = (
            chunk["traffic_label"]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({
                "1": 1,
                "attack": 1,
                "attack_pcaps": 1
            })
            .fillna(1)
            .astype("int32")
            .values
        )

    if "timestamp" in chunk.columns:
        ts = pd.to_datetime(chunk["timestamp"], errors="coerce")
        chunk["hour"] = ts.dt.hour.fillna(0).astype("int8")
        chunk["minute"] = ts.dt.minute.fillna(0).astype("int8")
        chunk["second"] = ts.dt.second.fillna(0).astype("int8")

    if "raw_log" in chunk.columns:
        chunk["log_length"] = chunk["raw_log"].fillna("").astype(str).str.len().astype("int32")
    elif "log_length" not in chunk.columns:
        chunk["log_length"] = 0

    for col in ["application_name", "application_category", "device_type"]:
        if col in chunk.columns:
            chunk[col] = safe_map_encoder(chunk[col], encoders[col], fallback_value=0)
        else:
            chunk[col] = 0

    drop_cols = [
        "timestamp", "src_ip", "dst_ip", "raw_log",
        "attack_type", "traffic_label", "line_number"
    ]
    chunk_model = chunk.drop(columns=[c for c in drop_cols if c in chunk.columns], errors="ignore")

    chunk_model = chunk_model.reindex(columns=feature_names, fill_value=0)
    chunk_model = chunk_model.replace([np.inf, -np.inf], np.nan).fillna(0)
    chunk_model = downcast_numeric(chunk_model)

    X_scaled = scaler.transform(chunk_model.values.astype("float32"))
    X_sel = selector.transform(X_scaled)

    if use_pca and pca is not None:
        X_final = pca.transform(X_sel)
    else:
        X_final = X_sel

    X_final = X_final.astype("float32")

    del chunk_model, X_scaled, X_sel
    gc.collect()

    return X_final, meta, y_true

Celda 5. Funcion de inferencia del IDS hibrido

In [10]:
def predict_hybrid_chunk(X_final: np.ndarray):
    xgb_prob = xgb.predict_proba(X_final)[:, 1]
    lgbm_prob = lgbm.predict_proba(X_final)[:, 1]
    dnn_prob = final_dnn.predict(X_final, batch_size=BATCH_PRED, verbose=0).reshape(-1)

    if use_autoencoder and autoencoder is not None:
        X_hat = autoencoder.predict(X_final, batch_size=BATCH_PRED, verbose=0)
        ae_score = np.mean(np.square(X_final - X_hat), axis=1).astype("float32")
        ae_bin = (ae_score > float(ae_threshold)).astype("int32")
        meta_features = np.column_stack([xgb_prob, lgbm_prob, dnn_prob, ae_score]).astype("float32")
    else:
        ae_score = None
        ae_bin = None
        meta_features = np.column_stack([xgb_prob, lgbm_prob, dnn_prob]).astype("float32")

    meta_features = meta_scaler.transform(meta_features)

    hybrid_prob = meta_learner.predict_proba(meta_features)[:, 1]
    hybrid_pred = (hybrid_prob >= 0.5).astype("int32")

    return {
        "xgb_prob": xgb_prob,
        "lgbm_prob": lgbm_prob,
        "dnn_prob": dnn_prob,
        "ae_score": ae_score,
        "ae_bin": ae_bin,
        "hybrid_prob": hybrid_prob,
        "hybrid_pred": hybrid_pred
    }

Celda 6. Procesamiento por chunks del fused_it_ot.csv

In [ ]:
def process_fused_it_ot(input_file: str):
    alerts_list = []
    acc = {
        "XGBoost": {"tn": 0, "fp": 0, "fn": 0, "tp": 0},
        "LightGBM": {"tn": 0, "fp": 0, "fn": 0, "tp": 0},
        "DNN": {"tn": 0, "fp": 0, "fn": 0, "tp": 0},
        "Hybrid": {"tn": 0, "fp": 0, "fn": 0, "tp": 0},
    }

    if use_autoencoder and autoencoder is not None:
        acc["Autoencoder"] = {"tn": 0, "fp": 0, "fn": 0, "tp": 0}
    
    latencias_chunk = []
    latencias_instancia = []

    for chunk in pd.read_csv(input_file, chunksize=CHUNK_SIZE, low_memory=False):
        try:
            chunk = downcast_numeric(chunk)
            inicio = time.perf_counter()
            X_final, meta, y_true = preprocess_fused_it_ot_chunk(chunk)

            if X_final is None:
                del chunk
                gc.collect()
                continue

            preds = predict_hybrid_chunk(X_final)
            fin = time.perf_counter()
            latencia_ms = (fin - inicio) * 1000
            latencia_por_instancia_ms = latencia_ms / len(X_final)
            latencias_chunk.append(latencia_ms)
            latencias_instancia.append(latencia_por_instancia_ms)
            y_pred_xgb = (preds["xgb_prob"] >= 0.5).astype("int32")
            y_pred_lgbm = (preds["lgbm_prob"] >= 0.5).astype("int32")
            y_pred_dnn = (preds["dnn_prob"] >= 0.5).astype("int32")
            y_pred_hybrid = preds["hybrid_pred"]

            if y_true is not None:
                for model_name, y_pred in [
                    ("XGBoost", y_pred_xgb),
                    ("LightGBM", y_pred_lgbm),
                    ("DNN", y_pred_dnn),
                    ("Hybrid", y_pred_hybrid),
                ]:
                    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
                    acc[model_name]["tn"] += int(cm[0, 0])
                    acc[model_name]["fp"] += int(cm[0, 1])
                    acc[model_name]["fn"] += int(cm[1, 0])
                    acc[model_name]["tp"] += int(cm[1, 1])

                if "Autoencoder" in acc and preds["ae_bin"] is not None:
                    cm_ae = confusion_matrix(y_true, preds["ae_bin"], labels=[0, 1])
                    acc["Autoencoder"]["tn"] += int(cm_ae[0, 0])
                    acc["Autoencoder"]["fp"] += int(cm_ae[0, 1])
                    acc["Autoencoder"]["fn"] += int(cm_ae[1, 0])
                    acc["Autoencoder"]["tp"] += int(cm_ae[1, 1])
            alert_mask = y_pred_hybrid == 1

            if alert_mask.any():
                meta_alert = meta.loc[alert_mask].reset_index(drop=True)

                alert_chunk = pd.DataFrame({
                    "timestamp": meta_alert["timestamp"] if "timestamp" in meta_alert.columns else np.nan,
                    "src_ip": meta_alert["src_ip"] if "src_ip" in meta_alert.columns else np.nan,
                    "dst_ip": meta_alert["dst_ip"] if "dst_ip" in meta_alert.columns else np.nan,
                    "device_type": meta_alert["device_type"] if "device_type" in meta_alert.columns else np.nan,
                    "application_name": meta_alert["application_name"] if "application_name" in meta_alert.columns else np.nan,
                    "application_category": meta_alert["application_category"] if "application_category" in meta_alert.columns else np.nan,
                    "hybrid_prob": preds["hybrid_prob"][alert_mask],
                    "hybrid_pred": y_pred_hybrid[alert_mask],
                    "xgb_prob": preds["xgb_prob"][alert_mask],
                    "lgbm_prob": preds["lgbm_prob"][alert_mask],
                    "dnn_prob": preds["dnn_prob"][alert_mask],
                })

                if preds["ae_score"] is not None:
                    alert_chunk["ae_score"] = preds["ae_score"][alert_mask]
                    alert_chunk["ae_bin"] = preds["ae_bin"][alert_mask]

                if y_true is not None:
                    y_true_alert = y_true[alert_mask]
                    alert_chunk["traffic_label_real"] = y_true_alert

                    if "attack_type" in meta_alert.columns:
                        alert_chunk["attack_type_real"] = meta_alert["attack_type"].values
                    else:
                        alert_chunk["attack_type_real"] = np.nan

                alerts_list.append(alert_chunk)

            del chunk, X_final, meta, preds
            

        except Exception as e:
            print("Error procesando chunk:", e)
            gc.collect()

    alerts_df = pd.concat(alerts_list, ignore_index=True) if len(alerts_list) > 0 else pd.DataFrame()

    def summarize(a):
        total = a["tn"] + a["fp"] + a["fn"] + a["tp"]
        if total == 0:
            return {
                "accuracy": 0.0,
                "precision": 0.0,
                "recall": 0.0,
                "f1": 0.0,
                "confusion_matrix": np.array([[0, 0], [0, 0]])
            }

        acc_v = (a["tn"] + a["tp"]) / total
        prec = a["tp"] / (a["tp"] + a["fp"] + 1e-9)
        rec = a["tp"] / (a["tp"] + a["fn"] + 1e-9)
        f1v = 2 * prec * rec / (prec + rec + 1e-9)

        return {
            "accuracy": acc_v,
            "precision": prec,
            "recall": rec,
            "f1": f1v,
            "confusion_matrix": np.array([[a["tn"], a["fp"]], [a["fn"], a["tp"]]])
        }

    metrics = {name: summarize(a) for name, a in acc.items()}

    return alerts_df, metrics, latencias_chunk, latencias_instancia

Celda 7. Ejecutar IDS sobre fused_it_ot.csv

In [19]:
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"No se encontro el archivo: {INPUT_FILE}")

alerts_df, metrics, latencias_chunk, latencias_instancia = process_fused_it_ot(INPUT_FILE)

print("Procesamiento terminado.")
print("Numero de alertas:", len(alerts_df))

Procesamiento terminado.
Numero de alertas: 3822000


Celda 8. Reporte de metricas

In [ ]:
latencias_chunk = np.array(latencias_chunk, dtype=float)
latencias_instancia = np.array(latencias_instancia, dtype=float)

reporte_latencia = pd.DataFrame([{
    "Latencia media por chunk (ms)": round(latencias_chunk.mean(), 4),
    "Latencia mediana por chunk (ms)": round(np.median(latencias_chunk), 4),
    "Latencia p95 por chunk (ms)": round(np.percentile(latencias_chunk, 95), 4),
    "Latencia media por instancia (ms)": round(latencias_instancia.mean(), 6),
    "Latencia mediana por instancia (ms)": round(np.median(latencias_instancia), 6)
}])

print("\nMetrica de latencia")
display(reporte_latencia)
def reporte_solo_ataques(cm):
    _, _, fn, tp = cm.ravel()

    ataques_detectados = tp
    ataques_no_detectados = fn
    recall = tp / (tp + fn + 1e-9)

    reporte = pd.DataFrame([{
        "Ataques Detectados (TP)": f"{ataques_detectados:,}",
        "Ataques No Detectados (FN)": f"{ataques_no_detectados:,}",
        "Recall de Ataques": f"{recall * 100:.4f}%" 
    }])

    return reporte


Metrica de latencia


,Latencia media por chunk (ms),Latencia mediana por chunk (ms),Latencia p95 por chunk (ms),Latencia media por instancia (ms),Latencia mediana por instancia (ms)
0,723.629,702.7,868.3237,0.015424,0.014153


Celda 9. Guardado de alertas y resultado

In [ ]:
print("Reporte de ataques detectados:")
# Usamos display en lugar de print en la misma línea para que se vea como tabla
display(reporte_solo_ataques(metrics["Hybrid"]["confusion_matrix"]))

Reporte de ataques detectados:


,Ataques Detectados (TP),Ataques No Detectados (FN),Recall de Ataques
0,"3,822,000",77,99.9980%


Celda 10. Mostrar alertas principales con IP y OT

In [ ]:
if len(alerts_df) > 0:
    cols = [c for c in [
        "timestamp", "src_ip", "dst_ip", "device_type",
        "application_name", "application_category", "hybrid_pred"
    ] if c in alerts_df.columns]

    display(alerts_df[cols].head(1000000))
else:
    print("No se generaron alertas.")

,timestamp,src_ip,dst_ip,device_type,application_name,application_category,hybrid_pred
0,2019-04-25 05:49:18.440,192.168.1.195,52.139.250.253,IoT_normal_fridge_2,TLS,Web,1
1,2019-04-25 05:49:26.475,192.168.1.6,239.255.255.250,IoT_normal_fridge_2,SSDP,System,1
2,2019-04-25 05:49:33.949,192.168.1.79,239.255.255.250,IoT_normal_fridge_2,SamsungSDP,Network,1
3,2019-04-25 05:49:34.051,192.168.1.1,224.0.0.1,IoT_normal_fridge_2,IGMP,Network,1
4,2019-04-25 05:49:42.961,192.168.1.79,192.168.1.255,IoT_normal_fridge_2,SamsungSDP,Network,1
...,...,...,...,...,...,...,...
999995,2019-04-26 04:18:55.238,192.168.1.38,192.168.1.194,IoT_normal_GPS_Tracker_3,HTTP,Web,1
999996,2019-04-26 04:17:52.979,192.168.1.38,192.168.1.184,IoT_normal_GPS_Tracker_3,TLS,Web,1
999997,2019-04-26 04:18:56.454,192.168.1.38,192.168.1.184,IoT_normal_GPS_Tracker_3,TLS,Web,1
999998,2019-04-26 04:19:11.644,192.168.1.34,192.168.1.152,IoT_normal_GPS_Tracker_3,HTTP,Web,1
